# Treinar o modelo da API Nexus-Chem no Colab

Gera o arquivo `modelo_nexus_chem.joblib` com a **base real** do projeto. Depois é só colocar esse arquivo na pasta `artifacts/` do repositório da API e fazer o deploy.

> As versões das bibliotecas precisam ser as mesmas do `requirements.txt` da API, senão o modelo não carrega no servidor. Se o Colab pedir para reiniciar a sessão depois da instalação, reinicie e siga da célula 2.

In [ ]:
# 1. Instala as mesmas versões usadas pela API
!pip install -q scikit-learn==1.9.1 pandas==3.0.5 numpy==2.5.3 joblib==1.6.0

In [ ]:
# 2. Baixa o código da API (troque pelo link do seu repositório)
!git clone https://github.com/SEU_USUARIO/nexus-chem-api.git
%cd nexus-chem-api

In [ ]:
# 3. Envie o CSV da base: dataset_projeto_integrador_nexus_chem.csv
from google.colab import files
import shutil
enviado = files.upload()
nome = next(iter(enviado))
shutil.move(nome, f'data/{nome}')
print('Base em data/' + nome)

In [ ]:
# 4. Treina (GridSearch do Random Forest + K-Means). Leva uns 30-60s
!python -m training.train --csv data/dataset_projeto_integrador_nexus_chem.csv

In [ ]:
# 5. Confere as métricas no teste
import json
meta = json.load(open('artifacts/modelo_nexus_chem.json'))
print(json.dumps({k: meta[k] for k in ['melhores_hiperparametros', 'f1_validacao_cruzada', 'metricas_teste', 'kmeans']}, indent=2, ensure_ascii=False))

In [ ]:
# 6. (Opcional) Testa a API aqui mesmo, sem servidor
!pip install -q fastapi==0.162.0 httpx
from fastapi.testclient import TestClient
from app.main import app
from app.schemas import EXEMPLO
with TestClient(app) as c:
    print(c.post('/predict', json=EXEMPLO).json())

In [ ]:
# 7. Baixa o modelo treinado -> coloque os dois arquivos em artifacts/ no repositório
files.download('artifacts/modelo_nexus_chem.joblib')
files.download('artifacts/modelo_nexus_chem.json')